# Exercise 3.2: Table Maintenance Procedures

Traditional databases run maintenance automatically — Postgres has `autovacuum`, MySQL's InnoDB compacts in the background. Iceberg separates storage (object store) from compute (Spark, Trino), so there's no always-on process to run maintenance. Instead, you schedule these procedures as periodic jobs or use a catalog service that automates them.

In this exercise, you'll learn how to maintain Apache Iceberg tables using the NYC Yellow Taxi dataset:
- **Compact data files**: Consolidate small files into larger ones
- **Compact manifests**: Reduce metadata overhead
- **Expire snapshots**: Clean up old table history
- **Remove orphan files**: Reclaim storage from failed operations

## Learning Objectives
- Identify when maintenance is needed
- Run data file compaction procedures
- Optimize metadata with manifest compaction
- Configure and run snapshot expiration
- Understand orphan file removal
- Monitor table health metrics

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TableMaintenance") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.maintenance")
print("Namespace 'maintenance' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June 2023**.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO -- skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nTaxi data ready in MinIO!")

## Part 1: Create a Table with a Small File Problem

We'll simulate a streaming ingestion pattern by writing June taxi data in many small batches, creating a large number of small files.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.maintenance.taxi_trips")

spark.sql("""
    CREATE TABLE polaris.maintenance.taxi_trips (
        VendorID INT,
        tpep_pickup_datetime TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        passenger_count LONG,
        trip_distance DOUBLE,
        fare_amount DOUBLE,
        tip_amount DOUBLE,
        total_amount DOUBLE
    )
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '10485760')
""")

print("Taxi trips table created!")

In [ ]:
june_df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet") \
    .select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "passenger_count", "trip_distance", "fare_amount",
            "tip_amount", "total_amount")

total_rows = june_df.count()
batches = 30
rows_per_batch = total_rows // batches

print(f"Writing {total_rows:,} rows in {batches} small batches ({rows_per_batch:,} rows each)...")

for i in range(batches):
    micro = june_df.limit(rows_per_batch)
    micro.writeTo("polaris.maintenance.taxi_trips").append()
    if (i + 1) % 10 == 0:
        print(f"  Created {i + 1}/{batches} commits...")

print("\nSmall files created!")

### Assess the Small File Problem

In [ ]:
print("File statistics BEFORE compaction:")
spark.sql("""
    SELECT
        COUNT(*) as file_count,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
        ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
        ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.maintenance.taxi_trips.files
""").show()

In [ ]:
snapshot_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

Many small files and many snapshots — this table needs maintenance!

## Part 2: Manifest Compaction

A manifest is an Avro file that lists a group of data files along with their partition values and column-level statistics (min, max, null count). During query planning, Iceberg reads manifests to decide which data files to scan. Too many small manifests means more files to open during planning — compaction merges them into fewer, larger manifests.

### Check Manifest Statistics

In [ ]:
print("Manifest statistics BEFORE compaction:")
spark.sql("""
    SELECT COUNT(*) as manifest_count, ROUND(AVG(length) / 1024, 2) as avg_size_kb
    FROM polaris.maintenance.taxi_trips.manifests
""").show()

### Compact Manifests

In [ ]:
print("Compacting manifests...")
start = time.time()

spark.sql("""
    CALL polaris.system.rewrite_manifests(table => 'maintenance.taxi_trips')
""")

elapsed = time.time() - start
print(f"Manifest compaction completed in {elapsed:.2f} seconds")

In [ ]:
print("Manifest statistics AFTER compaction:")
spark.sql("""
    SELECT COUNT(*) as manifest_count, ROUND(AVG(length) / 1024, 2) as avg_size_kb
    FROM polaris.maintenance.taxi_trips.manifests
""").show()

Fewer, larger manifest files means faster query planning.

## Part 3: Data File Compaction

### Compact Data Files

In [ ]:
print("Compacting data files...")
start = time.time()

spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'maintenance.taxi_trips',
        options => map('target-file-size-bytes', '10485760')
    )
""")

elapsed = time.time() - start
print(f"Data file compaction completed in {elapsed:.2f} seconds")

### Compare Before and After

> **Note on benchmarks:** The timings below illustrate the *relative* impact of compaction on query performance. Absolute numbers will vary based on your local environment. In production with larger datasets and distributed clusters, the improvements from compaction are typically more dramatic.

In [ ]:
print("File statistics AFTER compaction:")
spark.sql("""
    SELECT
        COUNT(*) as file_count,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
        ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
        ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.maintenance.taxi_trips.files
""").show()

Far fewer files, much closer to target size!

In [ ]:
print("Files per partition after compaction:")
spark.sql("""
    SELECT partition, COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.maintenance.taxi_trips.files
    GROUP BY partition ORDER BY partition
""").show()

### Try It: Compact with a Different Target Size

Re-run `rewrite_data_files` with a different `target-file-size-bytes` (e.g., `'5242880'` for 5 MB or `'52428800'` for 50 MB). Compare the resulting file count and average file size. How does the target size affect the number of files per partition?

In [ ]:
# my_target_size = '5242880'  # 5 MB — try '52428800' (50 MB) too

# spark.sql(f"""
#     CALL polaris.system.rewrite_data_files(
#         table => 'maintenance.taxi_trips',
#         options => map('target-file-size-bytes', '{my_target_size}')
#     )
# """)

# spark.sql("""
#     SELECT COUNT(*) as file_count,
#            ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
#            ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
#            ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb
#     FROM polaris.maintenance.taxi_trips.files
# """).show()

## Part 4: Snapshot Expiration

### View Snapshot History

In [ ]:
before_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Snapshots BEFORE expiration: {before_count}")

print("\nRecent snapshots:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM polaris.maintenance.taxi_trips.snapshots
    ORDER BY committed_at DESC LIMIT 5
""").show(truncate=False)

### Expire Old Snapshots

In [ ]:
print("Expiring old snapshots (keeping last 5)...")
start = time.time()

# We set older_than far in the future so it doesn't exclude any snapshots by age —
# retain_last does the actual filtering. This is a common pattern because
# expire_snapshots applies both conditions: older_than AND beyond retain_last.
spark.sql("""
    CALL polaris.system.expire_snapshots(
        table => 'maintenance.taxi_trips',
        older_than => TIMESTAMP '2099-12-31 23:59:59',
        retain_last => 5
    )
""")

elapsed = time.time() - start
print(f"Snapshot expiration completed in {elapsed:.2f} seconds")

In [ ]:
after_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Snapshots AFTER expiration: {after_count}")
print(f"Removed {before_count - after_count} snapshots")

Expiring snapshots also deletes unreferenced data files from storage.

## Part 5: Orphan File Removal

Orphan files occur when writes fail partway through. The data file gets written to storage but is never committed to metadata.

In [ ]:
orphan_key = 'maintenance/taxi_trips/data/orphan-file-12345.parquet'

s3_client.put_object(
    Bucket='warehouse',
    Key=orphan_key,
    Body=b"This is an orphan file that was never committed"
)

print(f"Created fake orphan file: s3://warehouse/{orphan_key}")

### Remove Orphan Files

The SQL procedure `remove_orphan_files` requires the `older_than` timestamp to be at least 24 hours in the past, a safety measure to prevent deleting files from concurrent writes that haven't committed yet. Since we just created our orphan file moments ago, we use the Iceberg Action API (via py4j) which allows us to bypass this restriction for demonstration purposes.

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql.functions import lit, to_timestamp

print("Removing orphan files...")
start = time.time()

# Iceberg requires older_than to be at least 24 hours from now to prevent
# accidentally deleting files from in-progress writes. We set it to 1 second
# in the future so our freshly-created orphan is eligible for removal.
# To bypass the 24-hour safety check, we use the Action API via py4j.
jvm = spark._jvm
table = jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(
    spark._jsparkSession, "polaris.maintenance.taxi_trips"
)
result = jvm.org.apache.iceberg.spark.actions.SparkActions.get(spark._jsparkSession) \
    .deleteOrphanFiles(table) \
    .olderThan(jvm.java.lang.System.currentTimeMillis() + 1000) \
    .execute()

elapsed = time.time() - start
print(f"Orphan file removal completed in {elapsed:.2f} seconds")

In [ ]:
try:
    s3_client.head_object(Bucket='warehouse', Key=orphan_key)
    print("Orphan file still exists — check the older_than parameter")
except:
    print("Orphan file was successfully removed!")

## Part 6: Monitoring Table Health

In [ ]:
print("=" * 60)
print("TABLE HEALTH REPORT: polaris.maintenance.taxi_trips")
print("=" * 60)

file_stats = spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.maintenance.taxi_trips.files
""").collect()[0]

print(f"\nData Files:")
print(f"  Count: {file_stats['file_count']}")
print(f"  Average Size: {file_stats['avg_size_mb']} MB")
print(f"  Total Size: {file_stats['total_size_mb']} MB")

manifest_count = spark.sql("""
    SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.manifests
""").collect()[0][0]
print(f"\nManifests: {manifest_count}")

snapshot_stats = spark.sql("""
    SELECT COUNT(*) as count, MAX(committed_at) as latest
    FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]
print(f"\nSnapshots: {snapshot_stats['count']}")
print(f"Latest: {snapshot_stats['latest']}")

record_count = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips").collect()[0][0]
print(f"\nRecords: {record_count:,}")
print("=" * 60)

### Try It: Run the Full Maintenance Sequence on Your Own Table

Create a new table, write data in small batches to produce many small files, then apply the full maintenance sequence in the recommended order: (1) compact manifests, (2) compact data files, (3) expire snapshots, (4) remove orphan files. Use the health report query above to monitor improvements at each step.

In [ ]:
# my_table = "polaris.maintenance.my_test_table"

# Step 0: Create a table with a small-file problem
# spark.sql(f"DROP TABLE IF EXISTS {my_table}")
# spark.sql(f"""
#     CREATE TABLE {my_table}
#     USING iceberg PARTITIONED BY (days(tpep_pickup_datetime))
#     AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet` LIMIT 0
# """)
# df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet")
# for i in range(20):
#     df.limit(5000).writeTo(my_table).append()

# Step 1: Compact manifests
# spark.sql(f"CALL polaris.system.rewrite_manifests(table => '{my_table.split('.', 1)[1]}')")

# Step 2: Compact data files
# spark.sql(f"CALL polaris.system.rewrite_data_files(table => '{my_table.split('.', 1)[1]}')")

# Step 3: Expire snapshots
# spark.sql(f"CALL polaris.system.expire_snapshots(table => '{my_table.split('.', 1)[1]}', retain_last => 2)")

# Step 4: Check the health report
# spark.sql(f"SELECT COUNT(*) as files FROM {my_table}.files").show()
# spark.sql(f"SELECT COUNT(*) as snapshots FROM {my_table}.snapshots").show()

# Cleanup
# spark.sql(f"DROP TABLE IF EXISTS {my_table}")

## Cleanup

In [ ]:
# Optional: Drop table
# spark.sql("DROP TABLE IF EXISTS polaris.maintenance.taxi_trips")
# print("Table dropped!")